# Stats 161 Midterm Project

## Data import

We select **12 consecutive weeks of Polymarket data, from June 1 to August 23, 2026**, to study wallet trading behavior. All dates and timestamps use UTC.


### 1. Setup


In [1]:
import pandas as pd
import requests

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from io import BytesIO


### 2. Dates and access settings


In [2]:
start_date = "2026-06-01"
end_date = "2026-08-23"
dates = pd.date_range(start_date, end_date, freq="D", tz="UTC")

base_url = "https://d1yyqbluk0oz0t.cloudfront.net"
course_password = 'PolymarketData2026!'


### 3. Wallet trades

We combine the 84 daily trade files into `wallet_trades`, keeping all wallets and all eight original variables. Each row is a recorded fill; one order may produce multiple fills.

| Variable | Description |
|---|---|
| `wallet` | Pseudonymous wallet ID, consistent across daily files. |
| `market` | Market slug; links to `markets.slug`. |
| `time` | Fill timestamp in UTC. |
| `side` | BUY or SELL for the recorded wallet. |
| `outcome` | Traded outcome, such as Yes, No, Up, Down, or a team name. |
| `price` | Execution price per share. |
| `size` | Number of shares traded. |
| `dollars` | Recorded dollar value of the fill. |


In [3]:
session = requests.Session()
session.auth = ("polymarketdatabase", course_password)
daily_trades = []

for day in dates:
    date_string = day.strftime("%Y-%m-%d")
    url = f"{base_url}/wallet_trades/trades-{date_string}.parquet"

    response = session.get(url, timeout=120)
    response.raise_for_status()
    day_trades = pd.read_parquet(BytesIO(response.content), dtype_backend="pyarrow")
    day_trades["time"] = pd.to_datetime(day_trades["time"], utc=True)

    # Check the file date.
    assert len(day_trades) > 0, f"Empty file: {date_string}"
    assert day_trades["time"].min().date() == day.date(), f"Unexpected date: {date_string}"
    assert day_trades["time"].max().date() == day.date(), f"Unexpected date: {date_string}"

    daily_trades.append(day_trades)
    if len(daily_trades) % 28 == 0:
        print(f"Loaded {len(daily_trades)} of {len(dates)} daily files.")

wallet_trades = pd.concat(daily_trades, ignore_index=True)
session.close()
del daily_trades, day_trades, response

print(f"Wallet trades imported: {len(wallet_trades):,} rows.")


Loaded 28 of 84 daily files.
Loaded 56 of 84 daily files.
Loaded 84 of 84 daily files.
Wallet trades imported: 66,736,398 rows.


### 4. Related market data

We import market, event, and settlement records for the markets in `wallet_trades`. All original columns are retained; key variables are listed below.

| DataFrame | Content | Key variables |
|---|---|---|
| `markets` | Market questions, outcome labels, and parent events. | `slug`, `title`, `event_slug`, `condition_id`, `outcomes`, `token_ids`, `start_date`, `end_date` |
| `events` | Event titles, descriptions, and scheduled dates. | `slug`, `title`, `description`, `start_date`, `end_date` |
| `resolutions` | Available settlement records. | `market_slug`, `resolved_yes`, `price_at_close`, `resolved_at` |

**Links:** `wallet_trades.market` → `markets.slug`; `markets.event_slug` → `events.slug`; `resolutions.market_slug` → `markets.slug`.

Market prices and volumes in the metadata reflect the current stored snapshot. Interpret resolution labels using the market's actual outcome labels.


In [4]:
database_url = URL.create(
    "postgresql+psycopg2",
    username="polymarketdatabase.merbqatlxbrcvtbnuaqe",
    password=course_password,
    host="aws-0-us-west-2.pooler.supabase.com",
    port=6543,
    database="postgres",
)

engine = create_engine(
    database_url,
    connect_args={"sslmode": "require", "connect_timeout": 20},
    execution_options={"postgresql_readonly": True},
)

market_slugs = wallet_trades["market"].dropna().unique().tolist()

with engine.connect() as connection:
    markets = pd.read_sql_query(
        text("SELECT * FROM public.markets WHERE slug = ANY(:slugs)"),
        connection,
        params={"slugs": market_slugs},
    )
    resolutions = pd.read_sql_query(
        text("SELECT * FROM public.resolutions WHERE market_slug = ANY(:slugs)"),
        connection,
        params={"slugs": market_slugs},
    )


In [5]:
event_slugs = markets["event_slug"].dropna().unique().tolist()

with engine.connect() as connection:
    events = pd.read_sql_query(
        text("SELECT * FROM public.events WHERE slug = ANY(:slugs)"),
        connection,
        params={"slugs": event_slugs},
    )

engine.dispose()


### 5. Import check

Confirm the row and column counts, and report traded markets without matching catalog metadata.


In [6]:
for name, data in [("wallet_trades", wallet_trades), ("markets", markets),
                   ("events", events), ("resolutions", resolutions)]:
    print(f"{name}: {len(data):,} rows, {len(data.columns)} columns")

missing_market_slugs = set(market_slugs) - set(markets["slug"])
print(f"Traded market slugs without catalog metadata: {len(missing_market_slugs):,}")
print(f"Study period: {start_date} through {end_date} (UTC)")


wallet_trades: 66,736,398 rows, 8 columns
markets: 227,655 rows, 25 columns
events: 98,132 rows, 13 columns
resolutions: 42,877 rows, 6 columns
Traded market slugs without catalog metadata: 119
Study period: 2026-06-01 through 2026-08-23 (UTC)


## Data Cleaning

### 1. Check data quality

Check missing values, trade fields, recorded amounts (tolerance: $0.01), repeated rows, and market links. Counts can overlap.


In [7]:
missing_values = wallet_trades.isna().any(axis=1)
invalid_side = ~wallet_trades["side"].isin(["BUY", "SELL"])
invalid_price = ~wallet_trades["price"].between(0, 1)
negative_size = wallet_trades["size"] < 0
negative_dollars = wallet_trades["dollars"] < 0
amount_mismatch = (
    wallet_trades["dollars"] - wallet_trades["price"] * wallet_trades["size"]
).abs() > 0.01
repeated_rows = wallet_trades.duplicated()
missing_metadata = ~wallet_trades["market"].isin(markets["slug"])

quality_check = pd.Series({
    "Missing values in any trade column": missing_values.sum(),
    "Side outside BUY / SELL": invalid_side.sum(),
    "Price outside [0, 1]": invalid_price.sum(),
    "Negative shares": negative_size.sum(),
    "Negative dollar value": negative_dollars.sum(),
    "Dollar difference above $0.01": amount_mismatch.sum(),
    "Repeated rows after the first": repeated_rows.sum(),
    "Trades without market metadata": missing_metadata.sum(),
}, name="Rows").to_frame()

quality_check


,Rows
Missing values in any trade column,0
Side outside BUY / SELL,0
"Price outside [0, 1]",0
Negative shares,0
Negative dollar value,0
Dollar difference above $0.01,0
Repeated rows after the first,2509300
Trades without market metadata,147668


### 2. Prepare the analysis data

Use `clean_trades` for the next analysis steps.

The checks found no missing or invalid trade values, so no rows need to be removed.

Repeated rows are kept because identical records cannot be confirmed as duplicates without a trade ID. Trades without market metadata are also kept. `has_market_metadata` indicates whether a trade's `market` matches `markets.slug`.

Large trades and low-activity wallets are retained.


In [8]:
clean_trades = wallet_trades.copy()
clean_trades["has_market_metadata"] = ~missing_metadata

print(f"Rows retained: {len(clean_trades):,}")
print(f"Rows removed: {len(wallet_trades) - len(clean_trades):,}")


Rows retained: 66,736,398
Rows removed: 0


## Wallet Features

Each row in `wallet_features` describes one wallet during a four-week period.

| Period | UTC dates |
|---|---|
| 1 | June 1–June 28, 2026 |
| 2 | June 29–July 26, 2026 |
| 3 | July 27–August 23, 2026 |

| Candidate feature | Definition |
|---|---|
| `active_days` | Number of days with at least one fill. |
| `fills_per_active_day` | Fill count divided by active days. |
| `median_fill_dollars` | Median recorded dollar value per fill. |
| `n_markets` | Number of distinct markets traded. |
| `top_market_share` | Largest market's share of recorded dollars; undefined if total dollars are zero. |
| `buy_fill_share` | BUY fills divided by all fills. |
| `buy_sell_market_share` | Share of traded markets with both BUY and SELL fills for at least one common outcome. |
| `busiest_day_share` | Busiest day's fill count divided by all fills. |
| `median_buy_price` | Median execution price among BUY fills; missing when there are no BUY fills. |

`wallet` and `period` identify each row. `n_fills` and `total_dollars` are supporting measures.


### 1. Four-week periods


In [9]:
feature_trades = clean_trades.copy()
feature_trades["day"] = feature_trades["time"].dt.floor("D")
feature_trades["period"] = (
    (feature_trades["day"] - pd.Timestamp(start_date, tz="UTC")).dt.days // 28 + 1
)
feature_trades["is_buy"] = feature_trades["side"] == "BUY"

group_keys = ["wallet", "period"]


### 2. Basic wallet activity


In [10]:
wallet_features = feature_trades.groupby(group_keys).agg(
    n_fills=("wallet", "size"),
    total_dollars=("dollars", "sum"),
    active_days=("day", "nunique"),
    median_fill_dollars=("dollars", "median"),
    n_markets=("market", "nunique"),
)

wallet_features["fills_per_active_day"] = (
    wallet_features["n_fills"] / wallet_features["active_days"]
)


### 3. Concentration across markets and days


In [11]:
market_dollars = feature_trades.groupby(group_keys + ["market"])["dollars"].sum()
top_market_dollars = market_dollars.groupby(level=group_keys).max()
wallet_features["top_market_share"] = (
    top_market_dollars
    / wallet_features["total_dollars"].where(wallet_features["total_dollars"] > 0)
)

daily_fills = feature_trades.groupby(group_keys + ["day"]).size()
wallet_features["busiest_day_share"] = (
    daily_fills.groupby(level=group_keys).max() / wallet_features["n_fills"]
)


### 4. Buying and selling


In [12]:
wallet_features["buy_fill_share"] = feature_trades.groupby(group_keys)["is_buy"].mean()
feature_trades["buy_price"] = feature_trades["price"].where(feature_trades["is_buy"])
wallet_features["median_buy_price"] = (
    feature_trades.groupby(group_keys)["buy_price"].median()
)

outcome_sides = feature_trades.groupby(
    group_keys + ["market", "outcome"]
)["side"].nunique()
market_has_buy_sell = outcome_sides.eq(2).groupby(
    level=group_keys + ["market"]
).any()
wallet_features["buy_sell_market_share"] = (
    market_has_buy_sell.groupby(level=group_keys).sum() / wallet_features["n_markets"]
)


### 5. Wallet feature table

Keep every wallet-period with observed trades. A wallet with no trades in a period has no row for that period. Activity thresholds, scaling, and model selection come later.


In [13]:
feature_columns = [
    "active_days", "fills_per_active_day", "median_fill_dollars",
    "n_markets", "top_market_share", "buy_fill_share",
    "buy_sell_market_share", "busiest_day_share", "median_buy_price",
]
wallet_features = wallet_features.reset_index()[
    ["wallet", "period"] + feature_columns + ["n_fills", "total_dollars"]
]

print(f"Wallet-period rows: {len(wallet_features):,}")
print(f"Distinct wallets: {wallet_features['wallet'].nunique():,}")
print(f"Missing BUY-price medians: {wallet_features['median_buy_price'].isna().sum():,}")
wallet_features.head(6)


Wallet-period rows: 1,154,562
Distinct wallets: 779,519
Missing BUY-price medians: 67,918


,wallet,period,active_days,fills_per_active_day,median_fill_dollars,n_markets,top_market_share,buy_fill_share,buy_sell_market_share,busiest_day_share,median_buy_price,n_fills,total_dollars
0,0x000025e0e96474e2485e259558e53f2dd4a54626,1,4,12.0,42.349999,35,0.099013,0.729167,0.371429,0.395833,0.52,48,2188.836554
1,0x000028713780b8882e3f11559f9c173802b3d716,1,14,33.5,16.6907,223,0.033329,0.703625,0.578475,0.204691,0.33,469,10510.332006
2,0x000028713780b8882e3f11559f9c173802b3d716,2,1,2.0,6.999999,2,0.5,1.0,0.000000,1.0,0.152886,2,13.999999
3,0x0000381a4e54c779318b9005f6c637cafb631e62,1,22,2.272727,56.43,31,0.087263,0.7,0.451613,0.1,0.54,50,2735.1852
4,0x00006123754dc778ea3abdd86e71a5d6d8fbd5b1,2,1,1.0,37.229999,1,1.0,1.0,0.000000,1.0,0.41,1,37.229999
5,0x000096d4e7bfabe3e83b61bca461a83d082295df,1,2,1.0,1.7675,2,0.603961,1.0,0.000000,0.5,0.4485,2,3.534999
